# The Annotated Transformer

References
- The annotated transformer blogpost by Harvard NLP: https://nlp.seas.harvard.edu/annotated-transformer/
- Attention is All You Need

## The Paper

### Overview

The paper specifically investigates the sequence transduction models which typically follow an encoder-decoder architecture based on complex recurrent or convolutional neural networks. The paper proposes a (later-found-to-be) revolutionary network architecture called the **Transformer**, which based solely on attention mechanisms and delivers SOTA performance on machine translation tasks and also shows to generalize well to other tasks.

### The Original Goal

Sequence modeling and transduction problems such as language modeling and machine translation, which typically rely on recurrent models and encoder-decoder architectures to achieve SOTA results.

### What Are Recurrent Models?

Recurrent models generate a sequence of hidden states $h_t$ as a function of the previous hidden state $h_{t - 1}$ and the input for position $t$, which has an inherently sequential nature that makes it hard for parallelization and suffers memory issues from this fundamental constraint of sequential computation.

### How Does Attention Help?

The attention mechanism has been providing concrete improvement in sequence modeling and transduction models as they allow modeling of dependencies without regard to their distance in the input or output sequences.

### The Architecture

The transformer model proposed also follows a basic encoder-decoder architecture. Specifically, the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$. Then given the representations $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time in a *autoregressive* manner, which consumes the previously generated symbols as an additional input while generating the next. The transformer follows this architecture by using *stacked* self-attention and pointwise, fully connected layers for both the encoder and the decoder, as shown in the diagram below.
The transformer model proposed also follows a basic encoder-decoder architecture. Specifically, the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$. Then given the representations $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time in a *autoregressive* manner, which consumes the previously generated symbols as an additional input while generating the next. The transformer follows this architecture by using *stacked* self-attention and pointwise, fully connected layers for both the encoder and the decoder, as shown in the diagram below.

![Transformer architecture](imgs/transformer-arch.png)

#### The Encoder and the Decoder Stacks

##### Encoder

The encoder network is composed of a stack of $N = 67$ identical layers. Each layers has two sublayers:
- The first layer is a Multi-Head Self-Attention Mechanism
- The second layer is a simple, positionwise fully connected feed-forward network.
The encoder network also incorporates a residual connection around each two sublayers, followed by layer normalization, which means that the output of each sublayer is in the format,
$$\mathtt{LayerNorm}(x + \mathtt{SubLayer}(x)),$$
where $\mathtt{SubLayer}(x)$ represents the sublayer function iteself. To facilitate these residual connections, we keep all sublayers in the model, as well as the embedding layers produce outputs of the same dimension $d_\texttt{model} = 512$.

##### Decoder

The decoder network is also composed of a stack of $N = 6$ identical layers. In addition to the two sublayers in each encoder layer, the decoder adds in a third sublayer, which performs Multi-Head Attention over the output of the encoder stack. We also incorporate residual connections around each of the sublayers followed by the layer normalization as in the encoder network. Furthermore, we modify the self-attention sublayer in the decoder stack with causality masking to prevent attending to future tokens during generation, ensure that the predictions for position $i$ depend only on the previously known outputs at positions before $i$.

## The Code

In [ ]:
# required imports
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import DataLoader
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSEampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

warnings.filterwarnings("ignore")
RUN_EXAMPLES = True # set to False to skip example runs